In [78]:
# !pip install streamlit pyngrok
# !pip install PyPDF2

from openai import OpenAI
import os
import subprocess
import streamlit as st
import PyPDF2
from pyngrok import ngrok, conf

# Set OpenAI API key
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY_GOES_HERE"

# Writes Streamlit app to app.py file
app_code = """
import streamlit as st
import PyPDF2
import openai
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Initialize session state
if 'text' not in st.session_state or st.session_state.text is None:
    st.session_state.text = ""
if 'memory' not in st.session_state or st.session_state.memory is None:
    st.session_state.memory = []
if 'suggested_questions' not in st.session_state:
    st.session_state.suggested_questions = []


#This function extracts text from a PDF file
def extract_text_from_pdf(uploaded_file):
    pdf_reader = PyPDF2.PdfReader(uploaded_file)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text() or ""
    return text

#This function asks the agent a question and returns an answer
def ask_agent(memory, text, question):
    if not text:
        text = "[No document text available]"

    # Ensure memory is not empty
    if not memory:
        memory.append({"role": "system", "content": "You are a helpful assistant."})

    # Add the user's current question
    memory.append({
        "role": "user",
        "content": "Document Text: " + text[:3000] + "\\nQuestion: " + question
    })

    # AI call
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=memory,
        temperature=0.5,
        max_tokens=300
    )

    answer = response.choices[0].message.content.strip()
    memory.append({"role": "assistant", "content": answer})
    return answer


# This function generates proactive suggestions based on the text
def generate_proactive_suggestions(text):
    if not text:
        return [], "[Upload a PDF Document to summarize]"

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": (
                        "You are an AI that summarizes research documents and generates insightful follow-up questions.\\n\\n"
                        "Summarize the following document in 3–4 sentences. Then generate 4 highly relevant research questions.\\n\\n"
                        + text[:9000]
                    )
                }
            ],
            temperature=0.4,
            max_tokens=500
        )
        output = response.choices[0].message.content.strip()
    except Exception as e:
        output = "[Error generating summary/questions: {}]".format(str(e))

    # Parse model output into summary + questions
    if "Questions:" in output:
        parts = output.split("Questions:")
        summary = parts[0].replace("Summary:", "").strip()
        questions_block = parts[1].strip()
        questions = [
            q.lstrip("- ").lstrip("0123456789. ").strip()
            for q in questions_block.split("\\n")
            if q.strip()
        ]
    else:
        summary = output
        questions = []

    return questions, summary

st.title("PDF Research Assistant")

uploaded_file = st.file_uploader("Upload a PDF", type="pdf")

if uploaded_file is not None:
    st.session_state.text = extract_text_from_pdf(uploaded_file)
    st.success("PDF text extracted successfully!")

# Generate proactive summary & questions
suggestions, summary = generate_proactive_suggestions(st.session_state.text)
st.session_state.suggested_questions = suggestions

st.subheader("Research Summary")
st.write(summary)

st.subheader("Suggested Questions")
for q in suggestions:
    st.write(f"- {q}")

question = st.text_input("Enter your question:")

if st.button("Ask"):
    if question.strip() == "":
        st.warning("Please enter a question first.")
    else:
        answer = ask_agent(st.session_state.memory, st.session_state.text, question)
        st.write(answer)

        st.markdown("---")
        st.subheader("Conversation History")
        for msg in st.session_state.memory:
            if msg["role"] == "user":
                st.write(f"**You:** {msg['content'].split('Question:')[-1].strip()}")
            else:
                st.write(f"**Assistant:** {msg['content']}")
"""

# Write app.py to file
with open("app.py", "w") as f:
    f.write(app_code)

# Kill old ngrok instances
subprocess.run("pkill ngrok || true", shell=True)

# Start Streamlit to run app.py file on port 8502
streamlit_cmd = "streamlit run app.py --server.address=0.0.0.0 --server.port=8502"
process = subprocess.Popen(streamlit_cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Start ngrok tunnel
ngrok.set_auth_token("NGROK_AUTH_TOKEN_GOES_HERE")

try:
    agentic_ai_url = ngrok.connect(
        addr=8502,
        bind_tls=True,
        hostname="chronogrammatic-semicatalytic-tam.ngrok-free.dev"
    )
except Exception as e:
    print("NGROK ERROR:")
    print(e)
    raise SystemExit

agentic_ai_url


<NgrokTunnel: "https://chronogrammatic-semicatalytic-tam.ngrok-free.dev" -> "http://localhost:8502">